# Aula 01 — Ferramentas, Ambiente e Versionamento

**Aluno:** Davi Horácio de Souza · **Matrícula:** 20250059934

Neste exercício refaço o roteiro da aula 1: criar a estrutura de um projeto de dados, versioná-lo com Git, isolar o ambiente com um ambiente virtual e registrar as dependências.

Os comandos de terminal aparecem em blocos `bash`, cada um seguido da saída obtida ao rodá-lo e de uma análise do resultado. Eles foram executados na pasta `Unidade_01/`: a própria unidade é o projeto de vendas, e as próximas aulas continuam trabalhando nele.

## 1. Qual Python estou usando?

In [1]:
import sys

print(f"Versão do Python: {sys.version}")
print(f"Executável em uso: {sys.executable}")

Versão do Python: 3.13.13 | packaged by Anaconda, Inc. | (main, Apr 14 2026, 06:19:41) [GCC 14.3.0]
Executável em uso: /home/davi/Área de trabalho/Ciência de dados/ciencia-de-dados-ufrn/.venv/bin/python


**Análise:** o notebook roda com Python 3.13, e o executável fica dentro da pasta `.venv` deste repositório, não no Python do sistema. Isso confirma que as bibliotecas usadas aqui vêm de um ambiente isolado, criado só para esta disciplina. Se o caminho apontasse para `/usr/bin/python3`, qualquer pacote instalado afetaria o sistema inteiro e os outros projetos.

## 2. Ferramentas instaladas

```bash
git --version
uv --version
```

Saída:

```text
git version 2.53.0
uv 0.11.21 (x86_64-unknown-linux-gnu)
```

**Análise:** as duas ferramentas da aula estão disponíveis. O **Git** guarda o histórico do projeto e o **uv** cria o ambiente virtual e instala os pacotes. O uv faz o papel de `venv` + `pip` juntos e é bem mais rápido. Sem elas os passos seguintes não funcionariam, então essa checagem vem primeiro.

## 3. Nasce o projeto: estrutura de pastas

```bash
mkdir -p data src
ls
```

Saída:

```text
data
src
```

**Análise:** o projeto nasce com duas pastas, cada uma com um papel: `data/` guarda os dados brutos e `src/` o código reaproveitável. No roteiro também há uma pasta `notebooks/` para as análises. Aqui esse papel é das pastas `Aula_XX`, uma por aula, seguindo a estrutura pedida pelo professor. Separar isso desde o início evita misturar dados com código e facilita decidir o que deve ou não ir para o Git: os dados, por exemplo, ficam de fora.

## 4. Nasce a memória do projeto: Git e o primeiro commit

```bash
git init -b main
git config user.name
```

Saída:

```text
Repositório vazio Git inicializado em  /home/davi/Área de trabalho/Ciência de dados/ciencia-de-dados-ufrn/Unidade_01/.git/
Davi Horácio
```

**Análise:** o `git init` criou a pasta oculta `.git`, que é a "memória" do projeto: é ali que o histórico fica guardado. O `git config user.name` mostra o nome que vai assinar cada commit, então o histórico registra também *quem* fez cada mudança.

```bash
echo "# Projeto Vendas" > README.md
echo "Analise desenvolvida na disciplina de Ciencia de Dados." >> README.md
git status
```

Saída:

```text
No ramo main

No commits yet

Arquivos não monitorados:
  (utilize "git add <arquivo>..." para incluir o que será submetido)
	README.md

nada adicionado ao envio mas arquivos não registrados estão presentes (use "git add" to registrar)
```

**Análise:** o `README.md` aparece em "Arquivos não monitorados". O Git percebe que o arquivo existe, mas ainda não guarda nada. Versionar é uma ação explícita: só entra no histórico o que eu escolho com `git add`.

```bash
git add README.md
git commit -m "Adiciona README com a descricao do projeto"
git log --oneline
```

Saída:

```text
[main (root-commit) 2bfd740] Adiciona README com a descricao do projeto
 1 file changed, 2 insertions(+)
 create mode 100644 README.md
2bfd740 Adiciona README com a descricao do projeto
```

**Análise:** o primeiro commit (`root-commit`) registrou o README, e o `git log` agora mostra uma linha: um código curto que identifica o commit e a mensagem que escrevi. A partir daqui existe um ponto salvo ao qual o projeto sempre pode voltar.

## 5. Nasce o ambiente: isolamento com `uv venv`

```bash
uv venv --clear
ls -a
```

Saída:

```text
Using CPython 3.12.13
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate
.
..
.git
.venv
README.md
data
src
```

**Análise:** o `uv venv` criou o ambiente virtual na pasta `.venv`. Ela só aparece com `ls -a` porque o nome começa com ponto (arquivo oculto). É dentro dela que ficam o Python e os pacotes deste projeto, separados dos outros projetos da máquina.

```bash
git status
```

Saída:

```text
No ramo main
nothing to commit, working tree clean
```

**Análise:** no roteiro, este `git status` mostraria o "incidente": a pasta `.venv`, com milhares de arquivos, aparecendo como candidata a entrar no repositório. Aqui isso **não aconteceu**: a árvore está limpa. O próximo comando mostra por quê.

```bash
cat .venv/.gitignore
```

Saída:

```text
*
```

**Análise:** o próprio uv cria um `.gitignore` dentro do `.venv` com o padrão `*`, que manda o Git ignorar tudo naquela pasta. É uma proteção da ferramenta, mas não dá para depender dela: outras ferramentas, como o conda ou versões antigas do Python, não fazem isso. Também há outras coisas a ignorar além do ambiente. Por isso o projeto precisa do seu próprio `.gitignore`.

```bash
printf ".venv/\n__pycache__/\ndata/\n*.env\n.ipynb_checkpoints/\n" > .gitignore
git status
```

Saída:

```text
No ramo main
Arquivos não monitorados:
  (utilize "git add <arquivo>..." para incluir o que será submetido)
	.gitignore

nada adicionado ao envio mas arquivos não registrados estão presentes (use "git add" to registrar)
```

**Análise:** o `.gitignore` do projeto lista o que nunca deve ser versionado: o ambiente (`.venv/`), arquivos gerados pelo Python (`__pycache__/`, `.ipynb_checkpoints/`), os dados (`data/`, que podem ser grandes ou sensíveis) e segredos como senhas e chaves (`*.env`). O próprio `.gitignore` aparece como não monitorado: ele **deve** ser versionado, para que quem clonar o projeto herde as mesmas regras.

```bash
git add .gitignore
git commit -m "Adiciona .gitignore para ambiente, dados e segredos"
```

Saída:

```text
[main 111ac47] Adiciona .gitignore para ambiente, dados e segredos
 1 file changed, 5 insertions(+)
 create mode 100644 .gitignore
```

**Análise:** segundo commit registrado. Agora a proteção contra versionar o ambiente, os dados e os segredos faz parte do histórico do projeto.

## 6. O projeto ganha dependências

```bash
uv pip install pandas
uv pip list
```

Saída:

```text
Resolved 4 packages in 169ms
Installed 4 packages in 288ms
 + numpy==2.5.5
 + pandas==3.1.1
 + python-dateutil==4.0.0
 + tzdata==2026.6
Package         Version
--------------- -------
numpy           2.5.5
pandas          3.1.1
python-dateutil 4.0.0
tzdata          2026.6
```

**Análise:** o pandas foi instalado dentro do `.venv` do projeto, junto com as bibliotecas de que ele depende, como o numpy. O `uv pip list` mostra só esses poucos pacotes: o ambiente começou vazio e contém apenas o que o projeto realmente usa, sem nada herdado do sistema.

```bash
uv pip freeze > requirements.txt
cat requirements.txt
```

Saída:

```text
numpy==2.5.5
pandas==3.1.1
python-dateutil==4.0.0
tzdata==2026.6
```

**Análise:** o `uv pip freeze` gravou no `requirements.txt` cada pacote com a versão exata (`==`). O `.venv` não vai para o Git, então é esse arquivo que permite a outra pessoa, ou a mim em outro computador, recriar o mesmo ambiente com `uv pip install -r requirements.txt`. Fixar as versões evita que uma atualização do pandas mude os resultados da análise.

```bash
git add requirements.txt
git commit -m "Adiciona pandas e registra dependencias em requirements.txt"
```

Saída:

```text
[main b84211d] Adiciona pandas e registra dependencias em requirements.txt
 1 file changed, 4 insertions(+)
 create mode 100644 requirements.txt
```

**Análise:** terceiro commit. A lista de dependências está versionada e, se ela mudar no futuro, o Git vai mostrar exatamente qual pacote entrou ou mudou de versão.

## 7. Fechamento: o histórico conta a história

```bash
git log --oneline
```

Saída:

```text
b84211d Adiciona pandas e registra dependencias em requirements.txt
111ac47 Adiciona .gitignore para ambiente, dados e segredos
2bfd740 Adiciona README com a descricao do projeto
```

**Análise:** os três commits contam a evolução do projeto na ordem em que aconteceu: primeiro a descrição (README), depois a proteção (`.gitignore`) e por fim as dependências (`requirements.txt`). Com mensagens descritivas dá para entender o que foi feito sem abrir nenhum arquivo, e por isso vale fazer commits pequenos e frequentes.

> **Nota sobre este repositório:** um repositório Git dentro de outro não é enviado ao GitHub (aparece só como uma pasta vazia). Por isso, depois de seguir o roteiro, removi o `.git` interno da `Unidade_01/`, que ficou como uma pasta comum. O versionamento passa a ser feito pelo repositório da disciplina, onde cada aula entra como um commit. O `.venv` e a pasta `data/` continuam fora do Git, ignorados pelo `.gitignore`.

## Perguntas para consolidar

**1. Um colega cético diz que ambiente virtual é exagero para quem trabalha sozinho num projeto só. Ele tem razão?**

Não. Mesmo sozinho, ao longo do curso vou ter vários projetos, e eles podem precisar de versões diferentes da mesma biblioteca: atualizar o pandas para um projeto novo pode quebrar um antigo. O ambiente virtual, junto com o `requirements.txt`, também documenta do que o projeto precisa e permite reproduzi-lo em outro computador ou para o professor avaliar. O custo é um comando (`uv venv`), e o benefício é não ter surpresas.

**2. Se o arquivo `data/vendas.csv` for apagado, o Git consegue recuperá-lo?**

Não. Como a pasta `data/` está no `.gitignore`, o Git nunca guardou esse arquivo, então não há o que recuperar. Ignorar um arquivo evita que ele vá para o repositório, mas não o protege. Por isso o README documenta a fonte e as instruções de download dos dados: assim eles sempre podem ser baixados de novo. Se forem dados próprios, precisam de um backup separado.

**3. Um colega clona o repositório e não encontra a pasta `.venv`. Quais passos reproduzem o ambiente?**

O `.venv` não vem junto porque está no `.gitignore`. Para recriá-lo:

```bash
cd Unidade_01
uv venv
source .venv/bin/activate
uv pip install -r requirements.txt
```

Depois disso basta baixar os dados seguindo as instruções do README.

## Conclusão

O projeto saiu do zero com três decisões que se reforçam: **estrutura** (cada coisa na sua pasta), **versionamento** (cada passo registrado num commit) e **isolamento** (um ambiente próprio, descrito no `requirements.txt`). Juntas, elas permitem que qualquer pessoa clone o projeto e chegue ao mesmo ambiente e aos mesmos resultados, que é a base da reprodutibilidade em ciência de dados.